# 0. Install dependancies

- tensorflow==2.3.0
- gym
- keras
- keras-rl2

1. Test Random Env with openai gym

In [1]:
import gym
import random

In [2]:
env = gym.make('CartPole-v0')
states = env.observation_space.shape[0]
actions = env.action_space.n

In [3]:
actions

2

In [4]:
episodes = 10
for episode in range(1,episodes+1):
    state = env.reset()
    done=False
    score = 0

    while not done:
        env.render()
        action = random.choice([0,1])
        n_state,reward,done,info = env.step(action)
        score+=reward
    print('Episode:{} Score:{}'.format(episode,score))

Episode:1 Score:22.0
Episode:2 Score:13.0
Episode:3 Score:18.0
Episode:4 Score:46.0
Episode:5 Score:58.0
Episode:6 Score:18.0
Episode:7 Score:12.0
Episode:8 Score:19.0
Episode:9 Score:11.0
Episode:10 Score:13.0


2. Create a deep learning model with keras

In [5]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.optimizers import Adam


In [6]:
def build_model(states,actions):
    model = Sequential()
    model.add(Flatten(input_shape=(1,states)))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(actions,activation='linear'))
    return model



In [7]:
model = build_model(states,actions)
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
flatten (Flatten)            (None, 4)                 0         
_________________________________________________________________
dense (Dense)                (None, 24)                120       
_________________________________________________________________
dense_1 (Dense)              (None, 24)                600       
_________________________________________________________________
dense_2 (Dense)              (None, 2)                 50        
Total params: 770
Trainable params: 770
Non-trainable params: 0
_________________________________________________________________


3. Build agent with Keras-RL

In [8]:
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory


In [9]:
def build_agent(model,actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000,window_length=1)
    dqn = DQNAgent(model=model,memory=memory,policy=policy,
                   nb_actions=actions,nb_steps_warmup=10,target_model_update=1e-2)
    return dqn

In [10]:
dqn = build_agent(model,actions)
dqn.compile(Adam(lr=1e-3),metrics=['mae'])
dqn.fit(env,nb_steps=50000,visualize=False,verbose=1)

AttributeError: 'Sequential' object has no attribute '_compile_time_distribution_strategy'

4. Reloading agent from memory